In [ ]:
import logging
from pathlib import Path

# Set up professional logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

def initialize_workspace(root_dir: str = "."):
    """
    Initializes a standardized directory structure for the ML lifecycle.
    
    Args:
        root_dir (str): The target directory. Defaults to the current working 
                        directory for maximum portability on GitHub.
    """
    # Define professional folder hierarchy
    directories = [
        "data/raw",
        "data/processed/train",
        "data/processed/test",
        "data/processed/validation",
        "models/checkpoints",
        "notebooks",
        "results/plots",
        "results/metrics"
    ]
    
    base_path = Path(root_dir).resolve()
    logging.info(f"Initializing project environment at: {base_path}")

    try:
        for folder in directories:
            # Create directories (including parents) if they don't exist
            path = base_path / folder
            path.mkdir(parents=True, exist_ok=True)
            logging.info(f"Directory verified: {folder}")
            
        logging.info("\n✅ SUCCESS: Project structure is ready for execution.")

    except PermissionError:
        logging.error("❌ Permission denied: Unable to create folders in this directory.")
    except Exception as e:
        logging.error(f"❌ An unexpected error occurred: {e}")

if __name__ == "__main__":
    initialize_workspace()

In [ ]:
import pandas as pd
import logging
from pathlib import Path

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

def load_and_verify_data(data_folder: str = "data"):
    """
    Loads historical and future datasets and performs a schema consistency check.
    
    Args:
        data_folder (str): Relative path to the folder containing CSV files.
                           Defaults to 'data' to match our project structure.
    """
    # 1. Dynamic Path Management (High Impact for Portability)
    base_path = Path(".").resolve() / data_folder
    
    file_hist = "extended_complete_Historical_5000_clean.csv"
    file_future = "mid_future_with_preds_clean.csv"
    
    path_hist = base_path / file_hist
    path_future = base_path / file_future

    try:
        # 2. Loading Data with explicit encoding and error handling
        logging.info(f"Loading Historical Data: {file_hist}")
        df_hist = pd.read_csv(path_hist, encoding="latin1")
        
        logging.info(f"Loading Future Scenario Data: {file_future}")
        df_future = pd.read_csv(path_future, encoding="latin1")

        # 3. Structural Summary
        logging.info(f"Historical Shape: {df_hist.shape} | Future Shape: {df_future.shape}")

        # 4. Professional Consistency Check
        hist_cols = set(df_hist.columns)
        future_cols = set(df_future.columns)

        if hist_cols == future_cols:
            logging.info("✅ SUCCESS: Dataset columns are perfectly aligned.")
        else:
            missing_in_future = hist_cols - future_cols
            missing_in_hist = future_cols - hist_cols
            
            if missing_in_future:
                logging.warning(f"Columns present only in Historical: {missing_in_future}")
            if missing_in_hist:
                logging.warning(f"Columns present only in Future: {missing_in_hist}")

        return df_hist, df_future

    except FileNotFoundError as e:
        logging.error(f"❌ Critical Error: Data file not found. Ensure files are in the '{data_folder}' folder.")
        logging.error(f"Details: {e}")
        return None, None
    except Exception as e:
        logging.error(f"❌ An unexpected error occurred during data loading: {e}")
        return None, None

if __name__ == "__main__":
    # Execute the loader
    df_historical, df_future = load_and_verify_data()

In [ ]:
import os
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

from catboost import CatBoostRegressor
from xgboost import XGBRegressor

# Configure Professional Logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# =====================================================
# 1. GLOBAL CONFIGURATION & PATHS
# =====================================================
def get_config():
    """Returns project configurations for better maintainability."""
    return {
        "results_dir": Path("./results/plots").resolve(),
        "data_dir": Path("./data").resolve(),
        "targets": ['EUI', 'IDD', 'OCI'],
        "features": [
            'Cooling_SP', 'Wall_U', 'Window_U', 'Floor_U', 'Roof_U',
            'Window_SHGC', 'Infiltration_rate', 'Sensible HR_Eff'
        ],
        "noise_level": 0.12,
        "xgb_feature_noise": 0.05
    }

# =====================================================
# 2. ANALYSIS & METRICS LOGIC
# =====================================================
def calculate_metrics(y_true, y_pred):
    """Calculates R2, MAE, and CV-RMSE metrics."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mean_val = np.mean(y_true)
    cv_rmse = (rmse / mean_val * 100) if mean_val != 0 else 0
    return r2, mae, cv_rmse

# =====================================================
# 3. HIGH-IMPACT VISUALIZATION (Publication Quality)
# =====================================================
def plot_model_performance(dataset_name, target, plot_data, save_path):
    """
    Generates a high-resolution grid comparing multiple models.
    Combines Histograms for distribution and Scatters for fit analysis.
    """
    plt.rcParams.update({
        'font.family': 'serif',
        'font.serif': ['Times New Roman'],
        'font.size': 14
    })

    fig = plt.figure(figsize=(24, 18))
    grid = gridspec.GridSpec(2, 3, wspace=0.25, hspace=0.35)
    
    models_list = list(plot_data.keys())
    colors = {'train': '#9b59b6', 'test': '#2ecc71'} # Purple & Green

    for i, model_name in enumerate(models_list):
        data = plot_data[model_name]
        ytr, ytr_p = data['y_train_true'], data['y_train_pred']
        yte, yte_p = data['y_test_true'], data['y_test_pred']
        
        inner = gridspec.GridSpecFromSubplotSpec(4, 4, subplot_spec=grid[i])
        ax_top = fig.add_subplot(inner[0, :])
        ax_main = fig.add_subplot(inner[1:, :])

        # --- Distribution Analysis (Top Histograms) ---
        bins = np.linspace(min(ytr.min(), yte.min()), max(ytr.max(), yte.max()), 45)
        ax_top.hist(np.concatenate([ytr, ytr_p]), bins=bins, density=True, alpha=0.5, 
                    color=colors['train'], histtype='stepfilled', label='Train Distribution')
        ax_top.hist(np.concatenate([yte, yte_p]), bins=bins, density=True, alpha=0.5, 
                    color=colors['test'], histtype='stepfilled', label='Test Distribution')
        ax_top.axis("off")
        ax_top.set_title(model_name, fontsize=16, fontweight='bold', pad=20)

        # --- Fit Analysis (Main Scatter) ---
        ax_main.scatter(ytr, ytr_p, alpha=0.2, color=colors['train'], s=60, label='Train Data')
        ax_main.scatter(yte, yte_p, alpha=0.45, color=colors['test'], s=60, edgecolors='w', label='Test Data')
        
        # Identity line and Trends
        lo, hi = ax_main.get_xlim()[0], ax_main.get_xlim()[1]
        ax_main.plot([lo, hi], [lo, hi], 'k:', lw=2, label='Ideal Fit')
        
        # Add Metrics Textbox
        r2_tr, mae_tr, cv_tr = calculate_metrics(ytr, ytr_p)
        r2_te, mae_te, cv_te = calculate_metrics(yte, yte_p)
        stats = (f"TRAIN R²: {r2_tr:.3f}\nTEST R²: {r2_te:.3f}\nCV-RMSE: {cv_te:.1f}%")
        ax_main.text(0.95, 0.05, stats, transform=ax_main.transAxes, ha='right', 
                     bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

        ax_main.set_xlabel("Measured (True)")
        ax_main.set_ylabel("Predicted")
        ax_main.grid(True, linestyle='--', alpha=0.5)

    fig.suptitle(f"Performance Analysis: {dataset_name} ({target})", fontsize=22, fontweight='bold')
    
    # Save in multiple formats for GitHub/LaTeX/Presentations
    fig_name = f"summary_{dataset_name}_{target}".replace(" ", "_").lower()
    plt.savefig(save_path / f"{fig_name}.png", dpi=300, bbox_inches='tight')
    plt.close()
    logging.info(f"✔ Exported summary plot for {target}")

# =====================================================
# 4. EXECUTION ENGINE
# =====================================================
def run_benchmarking():
    config = get_config()
    config['results_dir'].mkdir(parents=True, exist_ok=True)

    # Initialize Models
    regressors = {
        "CatBoost": CatBoostRegressor(iterations=1000, depth=4, learning_rate=0.03, verbose=0, random_state=42),
        "XGBoost": XGBRegressor(n_estimators=1000, max_depth=4, learning_rate=0.03, random_state=42),
        "Random Forest": RandomForestRegressor(n_estimators=500, max_depth=10, random_state=42),
        "SVR": SVR(kernel='rbf', C=10)
    }

    files = {
        "Historical": config['data_dir'] / "extended_complete_Historical_5000_clean.csv",
        "Mid-Future": config['data_dir'] / "mid_future_with_preds_clean.csv"
    }

    for name, path in files.items():
        if not path.exists():
            logging.error(f"File missing: {path}")
            continue

        df = pd.read_csv(path, encoding='latin1')
        # Clean column names (handling hidden spaces/encoding artifacts)
        df.columns = df.columns.str.replace(u'\xa0', ' ').str.strip()
        
        # Map target columns consistently
        df.rename(columns={c: c.split('-')[0] for c in df.columns if '-' in c}, inplace=True)

        X = df[config['features']].values

        for target in config['targets']:
            if target not in df.columns: continue
            
            logging.info(f"Processing {name} Scenario -> Target: {target}")
            y = df[target].values
            # Adding synthetic noise for robustness testing
            y += np.random.normal(0, np.std(y) * config['noise_level'], len(y))

            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
            
            scaler = StandardScaler()
            X_train_s = scaler.fit_transform(X_train)
            X_test_s = scaler.transform(X_test)

            plot_data = {}
            for m_name, model in regressors.items():
                model.fit(X_train_s, y_train)
                plot_data[m_name] = {
                    "y_train_true": y_train, "y_train_pred": model.predict(X_train_s),
                    "y_test_true": y_test, "y_test_pred": model.predict(X_test_s)
                }

            plot_model_performance(name, target, plot_data, config['results_dir'])

if __name__ == "__main__":
    run_benchmarking()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from catboost import CatBoostRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# =====================================================
# 1. CONFIGURATION & STYLE
# =====================================================
def get_tracking_config():
    return {
        "results_dir": Path("./results/plots/tracking").resolve(),
        "data_dir": Path("./data").resolve(),
        "targets": ['EUI', 'IDD', 'OCI'],
        "features": ['Cooling_SP', 'Wall_U', 'Window_U', 'Floor_U', 'Roof_U', 
                     'Window_SHGC', 'Infiltration_rate', 'Sensible HR_Eff'],
        "sample_limit": 300
    }

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'font.size': 16
})

# =====================================================
# 2. TRACKING VISUALIZATION (High Impact)
# =====================================================
def plot_prediction_tracking(y_true, y_pred, scenario, target, save_dir, limit=300):
    """
    Creates a temporal/sample-index plot showing the gap between AI and reality.
    """
    # Prepare small sample for clarity in visualization
    y_true_s = pd.Series(y_true[:limit]).reset_index(drop=True)
    y_pred_s = pd.Series(y_pred[:limit]).reset_index(drop=True)
    x = np.arange(len(y_true_s))

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    cv_rmse = (rmse / np.mean(y_true)) * 100

    fig, ax = plt.subplots(figsize=(20, 8))

    # Error Gap Fill (Visualizing residuals)
    ax.fill_between(x, y_true_s, y_pred_s, color='#e74c3c', alpha=0.2, label='Error Gap')
    
    # Lines
    ax.plot(x, y_true_s, color='#2c3e50', lw=2, label='Observed (Measured)')
    ax.plot(x, y_pred_s, color='#1abc9c', lw=2.5, ls='--', label='AI Prediction')

    # Metrics Box
    stats = f"RMSE: {rmse:.4f}\nCV-RMSE: {cv_rmse:.1f}%"
    ax.text(0.02, 0.95, stats, transform=ax.transAxes, fontsize=18, 
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

    ax.set_title(f"Model Tracking Performance: {target} ({scenario})", fontsize=22, pad=20)
    ax.set_xlabel(f"Sample Index (Top {limit} instances)")
    ax.set_ylabel(f"{target} Value")
    ax.legend(loc='upper right', frameon=True, shadow=True)
    ax.grid(axis='y', alpha=0.3)

    # Save
    save_dir.mkdir(parents=True, exist_ok=True)
    file_name = f"tracking_{scenario}_{target}".lower()
    plt.savefig(save_dir / f"{file_name}.png", dpi=300, bbox_inches='tight')
    plt.close()

# =====================================================
# 3. UNIFIED PROCESSING ENGINE
# =====================================================
def run_tracking_analysis():
    config = get_tracking_config()
    results = []

    # Map files to scenario names
    scenario_files = {
        "Baseline": config['data_dir'] / "extended_complete_Historical_5000_clean.csv",
        "Mid-Future": config['data_dir'] / "mid_future_with_preds_clean.csv"
    }

    # Initialize model
    model = CatBoostRegressor(iterations=1000, depth=6, verbose=0, random_state=42)

    for scenario, file_path in scenario_files.items():
        if not file_path.exists():
            logging.error(f"Missing file: {file_path}")
            continue
            
        df = pd.read_csv(file_path, encoding='latin1')
        df.columns = df.columns.str.strip()
        
        # Consistent Target Mapping
        df.rename(columns={c: c.split('-')[0] for c in df.columns if '-' in c}, inplace=True)

        for target in config['targets']:
            if target not in df.columns: continue
            
            logging.info(f"Tracking {target} for {scenario}...")
            X = df[config['features']]
            y = df[target]

            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
            
            scaler = StandardScaler()
            X_train_s = scaler.fit_transform(X_train)
            X_test_s = scaler.transform(X_test)

            model.fit(X_train_s, y_train)
            y_pred = model.predict(X_test_s)

            # Generate Visualization
            plot_prediction_tracking(y_test.values, y_pred, scenario, target, 
                                    config['results_dir'], config['sample_limit'])
            
            # Record Metrics
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))
            results.append({"Scenario": scenario, "Target": target, "RMSE": rmse})

    # Save Table
    pd.DataFrame(results).to_csv(config['results_dir'] / "tracking_metrics.csv", index=False)
    logging.info("🚀 All tracking plots and metrics successfully exported.")

if __name__ == "__main__":
    run_tracking_analysis()

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# ==========================================
# 1. STYLE & CONFIGURATION
# ==========================================
def set_publication_style():
    """Sets a clean, minimalist aesthetic for research plots."""
    plt.rcParams.update({
        'font.family': 'serif',
        'font.serif': ['Times New Roman'],
        'font.size': 14,
        'axes.titlesize': 18,
        'axes.labelsize': 16,
        'axes.linewidth': 1.2,
        'svg.fonttype': 'none',
        'figure.facecolor': '#FFFFFF'
    })

def get_distribution_config():
    return {
        "data_dir": Path("./data").resolve(),
        "save_dir": Path("./results/plots/distribution").resolve(),
        "palette": {"History": "#1CB0C8", "Future": "#F56631"},
        "targets": ['EUI', 'IDD', 'OCI']
    }

# ==========================================
# 2. CORE PLOTTING LOGIC
# ==========================================
def plot_distribution_comparison(df, config):
    """Generates a professional side-by-side Violin+Box plot."""
    set_publication_style()
    config["save_dir"].mkdir(parents=True, exist_ok=True)
    
    fig, axes = plt.subplots(1, 3, figsize=(22, 8))
    
    for ax, target in zip(axes, config["targets"]):
        # Minimalist Frame Setup
        ax.set_facecolor('#FBFBFB') 
        for spine in ax.spines.values():
            spine.set_edgecolor('#CCCCCC')
            spine.set_linewidth(1.2)

        ax.grid(axis='y', linestyle='--', alpha=0.4, color='#DDDDDD', zorder=0)

        # Violin Layer (Density Estimation)
        sns.violinplot(
            data=df, x='Dataset', y=target, palette=config["palette"],
            inner=None, width=0.7, linewidth=1.2, bw_adjust=0.6, ax=ax, zorder=2
        )

        # Boxplot Layer (Statistical Markers)
        # Overlaying boxplots manually for granular control over aesthetics
        for i, group in enumerate(['History', 'Future']):
            data_subset = df[df['Dataset'] == group][target].dropna()
            ax.boxplot(
                data_subset, positions=[i], widths=0.12, patch_artist=True,
                showmeans=True, manage_ticks=False,
                boxprops=dict(facecolor='#444444', color='black', linewidth=1.5, alpha=0.8, zorder=3),
                medianprops=dict(color='white', linewidth=2, zorder=5),
                meanprops=dict(marker='D', markerfacecolor='black', markeredgecolor='white', markersize=6, zorder=6),
                whiskerprops=dict(color='black', linewidth=1.2, zorder=3),
                capprops=dict(color='black', linewidth=1.2, zorder=3)
            )

        ax.set_title(target, fontweight='bold', pad=15)
        ax.set_xlabel("")
        ax.set_ylabel("Value", labelpad=10)

    # Global Legend Integration
    handles = [plt.Rectangle((0, 0), 1, 1, color=config["palette"][k]) for k in config["palette"]]
    fig.legend(handles, config["palette"].keys(), loc='upper center', ncol=2, 
               frameon=True, edgecolor='#DDDDDD', bbox_to_anchor=(0.5, 1.05))

    plt.tight_layout(pad=3.0)
    
    # Professional Export
    base_name = "distribution_analysis"
    plt.savefig(config["save_dir"] / f"{base_name}.png", dpi=300, bbox_inches='tight')
    plt.savefig(config["save_dir"] / f"{base_name}.svg", bbox_inches='tight')
    plt.close()
    logging.info(f"✔ Distribution plots saved to: {config['save_dir']}")

# ==========================================
# 3. EXECUTION
# ==========================================
def main():
    config = get_distribution_config()
    
    try:
        # Portability: Load from relative data path
        df_hist = pd.read_csv(config["data_dir"] / "extended_complete_Historical_5000_clean.csv", encoding='latin1')
        df_fut  = pd.read_csv(config["data_dir"] / "mid_future_with_preds_clean.csv", encoding='latin1')

        # Unified Renaming (DRY Principle)
        rename_map = {
            'EUI-Baseline': 'EUI', 'EUI-Mid-future': 'EUI',
            'IDD-Baseline': 'IDD', 'IDD-Mid-future': 'IDD',
            'OCI-Baseline': 'OCI', 'OCI-Mid-future': 'OCI'
        }
        df_hist.rename(columns=rename_map, inplace=True)
        df_fut.rename(columns=rename_map, inplace=True)

        df_hist['Dataset'] = 'History'
        df_fut['Dataset'] = 'Future'
        df_combined = pd.concat([df_hist, df_fut], ignore_index=True)

        plot_distribution_comparison(df_combined, config)
        
    except Exception as e:
        logging.error(f"❌ Failed during distribution analysis: {e}")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
import logging
import gc

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from bayes_opt import BayesianOptimization

# Configure logging for professional feedback
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# ==========================================
# 1. ENHANCED VISUALIZATION ENGINE
# ==========================================
def plot_optimization_results(y_train, pred_train, y_test, pred_test, scenario, target, mode, save_dir):
    """Generates a high-fidelity diagnostic plot for optimized models."""
    plt.rcParams.update({'font.family': 'serif', 'font.serif': ['Times New Roman'], 'font.size': 12})
    
    fig = plt.figure(figsize=(12, 12))
    gs = gridspec.GridSpec(2, 1, height_ratios=[1, 4], hspace=0.15)
    ax_top = fig.add_subplot(gs[0])
    ax_main = fig.add_subplot(gs[1])

    c_train, c_test = '#9b59b6', '#2ecc71'

    # Histogram Distribution
    bins = np.linspace(min(y_train.min(), y_test.min()), max(y_train.max(), y_test.max()), 45)
    ax_top.hist(np.concatenate([y_train, pred_train]), bins=bins, density=True, alpha=0.5, color=c_train, histtype='stepfilled', label='Train Distribution')
    ax_top.hist(np.concatenate([y_test, pred_test]), bins=bins, density=True, alpha=0.5, color=c_test, histtype='stepfilled', label='Test Distribution')
    ax_top.axis("off")
    ax_top.set_title(f"{scenario} | {target} | {mode}", fontsize=16, fontweight='bold', pad=20)

    # Scatter Fit
    ax_main.scatter(y_train, pred_train, alpha=0.2, color=c_train, s=60, label='Train Data')
    ax_main.scatter(y_test, pred_test, alpha=0.45, color=c_test, s=60, edgecolors='w', label='Test Data')
    
    # Statistics Box
    r2_te = r2_score(y_test, pred_test)
    rmse_te = np.sqrt(mean_squared_error(y_test, pred_test))
    stats_text = f"Test R²: {r2_te:.3f}\nTest RMSE: {rmse_te:.4f}"
    ax_main.text(0.95, 0.05, stats_text, transform=ax_main.transAxes, ha='right', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    ax_main.set_xlabel("True Values")
    ax_main.set_ylabel("AI Predicted Values")
    ax_main.grid(True, linestyle=':', alpha=0.6)
    
    save_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_dir / f"{mode}_{scenario}_{target}".lower().replace(" ", "_") + ".png", dpi=300, bbox_inches='tight')
    plt.close()

# ==========================================
# 2. OPTIMIZATION ENGINE
# ==========================================
class CatBoostOptimizer:
    def __init__(self, X_train, y_train, X_test, y_test):
        self.X_train, self.y_train = X_train, y_train
        self.X_test, self.y_test = X_test, y_test

    def evaluate(self, depth, learning_rate, iterations, l2_leaf_reg):
        """Evaluation function for Bayesian Optimization."""
        model = CatBoostRegressor(
            depth=int(depth),
            learning_rate=learning_rate,
            iterations=int(iterations),
            l2_leaf_reg=l2_leaf_reg,
            loss_function='RMSE',
            verbose=0,
            random_state=42,
            allow_writing_files=False
        )
        model.fit(self.X_train, self.y_train)
        preds = model.predict(self.X_test)
        rmse = np.sqrt(mean_squared_error(self.y_test, preds))
        return -rmse  # BO maximizes, so we return negative RMSE

def run_advanced_pipeline():
    # Setup Paths
    data_dir = Path("./data")
    base_results = Path("./results/plots/optimized_models")
    targets = ['EUI', 'IDD', 'OCI']
    features = ['Cooling_SP', 'Wall_U', 'Window_U', 'Floor_U', 'Roof_U', 'Window_SHGC', 'Infiltration_rate', 'Sensible HR_Eff']
    
    scenarios = {
        "Baseline": data_dir / "extended_complete_Historical_5000_clean.csv",
        "Mid-Future": data_dir / "mid_future_with_preds_clean.csv"
    }

    for scenario, path in scenarios.items():
        if not path.exists(): continue
        
        logging.info(f"🚀 Starting Optimization for Scenario: {scenario}")
        df = pd.read_csv(path, encoding='latin1')
        df.columns = df.columns.str.strip()
        df.rename(columns={c: c.split('-')[0] for c in df.columns if '-' in c}, inplace=True)

        for target in targets:
            if target not in df.columns: continue
            
            logging.info(f"--- Optimizing {target} ---")
            X = df[features]
            y = df[target] + np.random.normal(0, df[target].std() * 0.12, len(df[target])) # Noise injection
            
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
            scaler = StandardScaler()
            X_train_s = scaler.fit_transform(X_train)
            X_test_s = scaler.transform(X_test)

            # 1. Perform Bayesian Optimization
            opt_engine = CatBoostOptimizer(X_train_s, y_train, X_test_s, y_test)
            pbounds = {
                'depth': (4, 10),
                'learning_rate': (0.01, 0.1),
                'iterations': (1000, 2500),
                'l2_leaf_reg': (1, 15)
            }
            
            optimizer = BayesianOptimization(f=opt_engine.evaluate, pbounds=pbounds, random_state=42, verbose=0)
            optimizer.maximize(init_points=10, n_iter=20) # Adjusted iterations for demo balance
            
            best_params = optimizer.max['params']
            logging.info(f"🏆 Optimal Params for {target}: {best_params}")

            # 2. Train Final Optimized Model
            final_model = CatBoostRegressor(
                depth=int(best_params['depth']),
                learning_rate=best_params['learning_rate'],
                iterations=int(best_params['iterations']),
                l2_leaf_reg=best_params['l2_leaf_reg'],
                loss_function='RMSE', verbose=0, random_state=42
            )
            final_model.fit(X_train_s, y_train)
            
            # 3. Visualize & Cleanup
            plot_optimization_results(y_train, final_model.predict(X_train_s), 
                                      y_test, final_model.predict(X_test_s), 
                                      scenario, target, "BO_CatBoost", base_results)
            
            del final_model; gc.collect()

if __name__ == "__main__":
    run_advanced_pipeline()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import logging
import gc

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from bayes_opt import BayesianOptimization

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# ==========================================
# 1. STYLE & PATH CONFIGURATION
# ==========================================
def get_analysis_config():
    return {
        "data_dir": Path("./data").resolve(),
        "results_dir": Path("./results/plots/convergence").resolve(),
        "targets": ['EUI', 'IDD', 'OCI'],
        "features": ['Cooling_SP', 'Wall_U', 'Window_U', 'Floor_U', 'Roof_U', 
                     'Window_SHGC', 'Infiltration_rate', 'Sensible HR_Eff'],
        "noise_level": 0.12
    }

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'font.size': 14
})

# ==========================================
# 2. LEARNING CURVE VISUALIZATION
# ==========================================
def plot_convergence(evals_result, scenario, target, save_path):
    """
    Visualizes Training vs Validation RMSE to detect overfitting.
    """
    train_rmse = evals_result['learn']['RMSE']
    test_rmse = evals_result['validation']['RMSE']
    iterations = range(len(train_rmse))

    fig, ax = plt.subplots(figsize=(12, 7))
    
    # Colors (Consistent with previous sections)
    c_train, c_test = '#9b59b6', '#2ecc71'

    ax.plot(iterations, train_rmse, label='Training Loss', color=c_train, lw=2, alpha=0.8)
    ax.plot(iterations, test_rmse, label='Validation Loss', color=c_test, lw=2, ls='--')

    # Highlight Global Minimum
    min_rmse = min(test_rmse)
    min_idx = test_rmse.index(min_rmse)
    ax.scatter(min_idx, min_rmse, color='red', s=80, edgecolors='white', zorder=5, 
               label=f'Optimal Point (RMSE: {min_rmse:.4f})')

    ax.set_title(f"Convergence Analysis: {target} ({scenario})", fontsize=18, fontweight='bold', pad=15)
    ax.set_xlabel("Number of Estimators (Trees)")
    ax.set_ylabel("RMSE Score")
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(frameon=True, shadow=True)

    # Export
    save_path.mkdir(parents=True, exist_ok=True)
    file_name = f"convergence_{scenario}_{target}".lower().replace(" ", "_")
    plt.savefig(save_path / f"{file_name}.png", dpi=300, bbox_inches='tight')
    plt.close()

# ==========================================
# 3. OPTIMIZATION & TRAINING ENGINE
# ==========================================
def run_convergence_study():
    config = get_analysis_config()
    scenarios = {
        "Baseline": config['data_dir'] / "extended_complete_Historical_5000_clean.csv",
        "Mid-Future": config['data_dir'] / "mid_future_with_preds_clean.csv"
    }

    for name, path in scenarios.items():
        if not path.exists(): continue
        
        logging.info(f"🔄 Analyzing Convergence for: {name}")
        df = pd.read_csv(path, encoding='latin1')
        df.columns = df.columns.str.strip().str.replace(u'\xa0', ' ')
        df.rename(columns={c: c.split('-')[0] for c in df.columns if '-' in c}, inplace=True)

        for target in config['targets']:
            if target not in df.columns: continue
            
            # Data Prep with Noise Injection
            y = df[target] + np.random.normal(0, df[target].std() * config['noise_level'], len(df[target]))
            X_tr, X_te, y_tr, y_te = train_test_split(df[config['features']], y, test_size=0.2, random_state=42)
            
            scaler = StandardScaler()
            X_tr_s = scaler.fit_transform(X_tr)
            X_te_s = scaler.transform(X_te)

            # --- 1. Bayesian Optimization (Simplified Range) ---
            def bo_func(depth, lr, l2):
                m = CatBoostRegressor(depth=int(depth), learning_rate=lr, l2_leaf_reg=l2, 
                                      iterations=1500, verbose=0, random_state=42)
                m.fit(X_tr_s, y_tr)
                return -np.sqrt(mean_squared_error(y_te, m.predict(X_te_s)))

            optimizer = BayesianOptimization(f=bo_func, pbounds={'depth':(4,10), 'lr':(0.01,0.1), 'l2':(1,10)}, random_state=42, verbose=0)
            optimizer.maximize(init_points=10, n_iter=15)
            best = optimizer.max['params']

            # --- 2. Final Training with Metric Tracking ---
            model = CatBoostRegressor(
                depth=int(best['depth']), learning_rate=best['lr'], l2_leaf_reg=best['l2'],
                iterations=2000, eval_metric='RMSE', verbose=0, random_state=42
            )
            
            # Passing eval_set is the "secret sauce" for learning curves
            model.fit(X_tr_s, y_tr, eval_set=(X_test_s, y_te), use_best_model=True)
            
            # --- 3. Visualize Convergence ---
            plot_convergence(model.get_evals_result(), name, target, config['results_dir'])
            logging.info(f"✔ {target} convergence plot generated.")
            
            del model; gc.collect()

if __name__ == "__main__":
    run_convergence_study()

In [ ]:
import os
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pathlib import Path

# =====================================================
# 1. تنظیمات بصری و استایل حرفه‌ای
# =====================================================
def set_plot_style():
    plt.rcParams.update({
        'font.family': 'serif',
        'font.serif': ['Times New Roman'],
        'font.size': 12,
        'axes.labelweight': 'bold',
        'axes.titleweight': 'bold',
        'figure.dpi': 300
    })

# =====================================================
# 2. تابع اصلی رسم پلات‌های ترکیبی
# =====================================================
def plot_bo_convergence_grid(results_store, target_map, save_dir):
    """
    رسم یک شبکه ۲ در ۳ برای نمایش روند کاهش RMSE در دوره‌های مختلف.
    """
    set_plot_style()
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    # تعریف ساختار پلات (۲ ردیف برای زمان، ۳ ستون برای اهداف)
    rows_ordered = ["Historical", "Mid-Future"]
    cols_ordered = ["EUI", "IDD", "OCI"]
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharex=True)
    plt.subplots_adjust(wspace=0.18, hspace=0.12, bottom=0.15)

    for i, timeframe in enumerate(rows_ordered):
        for j, target in enumerate(cols_ordered):
            ax = axes[i, j]
            
            # استخراج داده‌ها از نتایج BO
            # فرض: results_store[timeframe][target] لیستی از RMSE هاست
            data = results_store.get(timeframe, {}).get(target, [])
            
            if data:
                style = target_map.get(target, {'color': 'blue', 'marker': 'o', 'linestyle': '--'})
                iterations = range(1, len(data) + 1)

                ax.plot(
                    iterations, data,
                    color=style['color'],
                    marker=style['marker'],
                    markersize=4,
                    linewidth=1.0,
                    linestyle=style['linestyle'],
                    alpha=0.85,
                    label=f"{target} RMSE"
                )

                ax.grid(True, linestyle='--', alpha=0.4)
                # نمایش گرید فقط برای محور Y جهت تمیزی بیشتر
                ax.yaxis.grid(True, alpha=0.3)
                ax.xaxis.grid(False)
                
                # تنظیمات Legend داخلی (اختیاری)
                ax.legend(loc='upper right', fontsize=9, frameon=True, framealpha=0.7)

            # برچسب زدن ردیف‌ها (فقط برای ستون اول)
            if j == 0:
                ax.set_ylabel(f"{timeframe}\nRMSE", fontsize=16, labelpad=15)

            # حذف تیک‌های محور X برای ردیف بالا جهت یکپارچگی
            if i == 0:
                ax.tick_params(axis='x', which='both', bottom=False, labelbottom=False)

    # افزودن متن‌های محوری کلی (Global Labels)
    fig.text(0.5, 0.08, 'Optimization Iterations', ha='center', fontsize=22, fontweight='bold')
    fig.suptitle("Bayesian Optimization Progress: RMSE Reduction Analysis", fontsize=24, fontweight='bold', y=0.96)

    # ایجاد راهنمای (Legend) پایین صفحه با لیبل‌های تمیز
    legend_elements = [
        Line2D([0], [0], color='blue',  marker='o', linestyle='--', lw=2, label='Energy Use Intensity (EUI)'),
        Line2D([0], [0], color='green', marker='s', linestyle='-',  lw=2, label='Indoor Daylighting (IDD)'),
        Line2D([0], [0], color='red',   marker='^', linestyle='-',  lw=2, label='Overall Comfort Index (OCI)')
    ]

    fig.legend(
        handles=legend_elements,
        loc='lower center',
        ncol=3,
        bbox_to_anchor=(0.5, 0.02),
        fontsize=15,
        frameon=False
    )

    # ذخیره‌سازی با فرمت‌های مختلف
    output_path = save_dir / "Combined_Optimization_Progress"
    plt.savefig(f"{output_path}.png", dpi=300, bbox_inches='tight')
    plt.savefig(f"{output_path}.svg", format='svg', bbox_inches='tight')
    
    print(f"✅ پلات ترکیبی با موفقیت در مسیر زیر ذخیره شد:\n{output_path}.png")
    plt.show()

# =====================================================
# 3. نمونه اجرا (Example Execution)
# =====================================================
# این بخش باید پس از اتمام حلقه‌های BO در اسکریپت اصلی فراخوانی شود.
# target_map = {
#    'EUI': {'color': 'blue', 'marker': 'o', 'linestyle': '--'},
#    'IDD': {'color': 'green', 'marker': 's', 'linestyle': '-'},
#    'OCI': {'color': 'red', 'marker': '^', 'linestyle': '-'}
# }
# plot_bo_convergence_grid(results_store, target_map, results_path)

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from catboost import CatBoostRegressor, Pool
from pathlib import Path
import gc

# ==========================================
# 1. تنظیمات بصری (Scientific Aesthetics)
# ==========================================
def set_shap_theme():
    plt.rcParams.update({
        'font.family': 'serif',
        'font.serif': ['Times New Roman'],
        'font.size': 14,
        'axes.labelsize': 16,
        'axes.titlesize': 18,
        'figure.dpi': 300
    })

# ==========================================
# 2. توابع تحلیلی و گرافیکی
# ==========================================
def plot_shap_summary(shap_values, X, scenario, target, save_dir):
    """رسم پلات نقطه‌ای SHAP برای نمایش جهت تاثیر ویژگی‌ها"""
    mean_abs_shap = np.mean(np.abs(shap_values), axis=0)
    idx = np.argsort(mean_abs_shap)
    
    plt.figure(figsize=(12, 8))
    # جیتر (Jitter) برای جلوگیری از همپوشانی نقاط
    for i, feature_idx in enumerate(idx):
        s_values = shap_values[:, feature_idx]
        feature_vals = X.iloc[:, feature_idx]
        # نرمال‌سازی برای رنگ‌بندی (Low to High)
        norm_vals = (feature_vals - feature_vals.min()) / (feature_vals.max() - feature_vals.min() + 1e-9)
        
        plt.scatter(s_values, np.full_like(s_values, i) + np.random.normal(0, 0.1, len(s_values)),
                    c=norm_vals, cmap='coolwarm', s=10, alpha=0.5, edgecolor='none')

    plt.yticks(range(len(idx)), [X.columns[i] for i in idx])
    plt.axvline(0, color='black', lw=1, alpha=0.5)
    plt.title(f"SHAP Impact: {target} ({scenario})")
    plt.xlabel("SHAP Value (Impact on Prediction)")
    
    # افزودن Colorbar
    sm = plt.cm.ScalarMappable(cmap='coolwarm', norm=plt.Normalize(vmin=0, vmax=1))
    cbar = plt.colorbar(sm, ax=plt.gca(), fraction=0.03, pad=0.04)
    cbar.set_label('Feature Value (Low to High)')
    
    plt.savefig(save_dir / f"shap_summary_{scenario}_{target}.png", bbox_inches='tight')
    plt.close()

# ==========================================
# 3. موتور پردازش SHAP
# ==========================================
def run_full_shap_analysis(scenarios, best_params_map):
    config = {
        "results_dir": Path("./results/shap_analysis"),
        "features": ['Cooling_SP', 'Wall_U', 'Window_U', 'Floor_U', 'Roof_U', 
                     'Window_SHGC', 'Infiltration_rate', 'Sensible HR_Eff']
    }
    config["results_dir"].mkdir(parents=True, exist_ok=True)
    set_shap_theme()

    for name, file_path in scenarios.items():
        if not os.path.exists(file_path): continue
        
        df = pd.read_csv(file_path, encoding='latin1')
        # پاکسازی نام ستون‌ها (حذف فواصل و کاراکترهای نامرئی)
        df.columns = df.columns.str.strip().str.replace(u'\xa0', ' ')
        df.rename(columns={c: c.split('-')[0] for c in df.columns if '-' in c}, inplace=True)

        for target, params in best_params_map.items():
            if target not in df.columns: continue
            
            print(f"🧬 Computing SHAP for {target} in {name}...")
            X = df[config["features"]]
            y = df[target]

            # بازسازی مدل بهینه
            model = CatBoostRegressor(**params, loss_function='RMSE', verbose=0, random_state=42)
            model.fit(X, y)

            # استخراج مقادیر SHAP به صورت Native (بسیار سریع)
            shap_values = model.get_feature_importance(Pool(X, label=y), type='ShapValues')[:, :-1]

            # رسم نمودارها
            plot_shap_summary(shap_values, X, name, target, config["results_dir"])
            
            # آزادسازی حافظه
            del model; gc.collect()

# --- نمونه اجرا ---
if __name__ == "__main__":
    scenarios = {"Baseline": "data/historical_data.csv", "Future": "data/future_data.csv"}
    # پارامترهای فرضی (جایگزین با خروجی BO)
    params = {"EUI": {'depth': 6, 'learning_rate': 0.05, 'iterations': 1000}} 
    # run_full_shap_analysis(scenarios, params)

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostRegressor
import gc
from pathlib import Path

# Pymoo Components
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.core.problem import ElementwiseProblem
from pymoo.termination import get_termination
from pymoo.optimize import minimize

# ==========================================
# 1. تنظیمات و پیکربندی خروجی
# ==========================================
BASE_PATH = Path(r"C:\Users\Alire\Desktop\Comparision ML")
RESULTS_DIR = BASE_PATH / "Results_Final_Optimization_v2"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "HISTORY": BASE_PATH / "extended_complete_Historical_5000_clean.csv",
    "FUTURE": BASE_PATH / "mid_future_with_preds_clean.csv"
}

# هایپرپارامترهای الگوریتم ژنتیک
POP_SIZE = 120
GENS = 600
FEATURES = ['Cooling_SP', 'Wall_U', 'Window_U', 'Floor_U', 'Roof_U', 
            'Window_SHGC', 'Infiltration_rate', 'Sensible HR_Eff']

plt.rcParams.update({'font.family': 'serif', 'font.size': 12})

# ==========================================
# 2. تعریف مسئله بهینه‌سازی (3-Objective)
# ==========================================
class BuildingOptimizationProblem(ElementwiseProblem):
    def __init__(self, models, scaler, xl, xu):
        super().__init__(n_var=len(FEATURES), n_obj=3, n_ieq_constr=0, xl=xl, xu=xu)
        self.models = models # [model_eui, model_idd, model_oci]
        self.scaler = scaler

    def _evaluate(self, x, out, *args, **kwargs):
        # استانداردسازی ورودی برای مدل‌های CatBoost
        x_scaled = self.scaler.transform(x.reshape(1, -1))
        
        # پیش‌بینی اهداف (EUI, IDD, OCI)
        f1 = self.models[0].predict(x_scaled)[0]
        f2 = self.models[1].predict(x_scaled)[0]
        f3 = self.models[2].predict(x_scaled)[0]
        
        out["F"] = [f1, f2, f3]

# ==========================================
# 3. تحلیل تصمیم‌گیری TOPSIS (با وزن‌دهی انتروپی)
# ==========================================

def apply_advanced_topsis(inputs, outputs):
    """
    رتبه‌بندی راهکارهای پارتو بر اساس کمترین فاصله از ایده‌آل مثبت
    """
    df = pd.DataFrame(inputs, columns=FEATURES)
    target_cols = ['EUI', 'IDD', 'OCI']
    df[target_cols] = outputs

    # ۱. نرمال‌سازی ماتریس تصمیم
    data = df[target_cols].values
    norm_data = (data - data.min(axis=0)) / (data.max(axis=0) - data.min(axis=0) + 1e-10)

    # ۲. محاسبه وزن‌ها به روش انتروپی (Entropy Weights)
    p = norm_data / (norm_data.sum(axis=0) + 1e-10)
    entropy = - (1/np.log(len(df)+1e-10)) * np.sum(p * np.log(p + 1e-10), axis=0)
    weights = (1 - entropy) / (1 - entropy).sum()
    
    print(f"📊 Entropy Weights: EUI={weights[0]:.2f}, IDD={weights[1]:.2f}, OCI={weights[2]:.2f}")

    # ۳. محاسبه فواصل از ایده‌آل مثبت و منفی
    weighted_norm = norm_data * weights
    best_ideal = weighted_norm.min(axis=0)  # چون هدف کمینه‌سازی است
    worst_ideal = weighted_norm.max(axis=0)

    dist_p = np.sqrt(((weighted_norm - best_ideal)**2).sum(axis=1))
    dist_n = np.sqrt(((weighted_norm - worst_ideal)**2).sum(axis=1))
    
    df['Performance_Score'] = dist_n / (dist_p + dist_n + 1e-10)
    return df.sort_values('Performance_Score', ascending=False).reset_index(drop=True)

# ==========================================
# 4. اجرای فرآیند بهینه‌سازی
# ==========================================
def run_full_pipeline(scenario_name, file_path):
    print(f"\n🚀 Starting Optimization for: {scenario_name}")
    
    # بارگذاری و تمیزکاری داده‌ها
    df = pd.read_csv(file_path, encoding='latin1')
    df.columns = df.columns.str.strip().str.replace(u'\xa0', ' ')
    df.rename(columns={c: c.split('-')[0] for c in df.columns if '-' in c}, inplace=True)

    # آماده‌سازی مدل‌های جایگزین (Surrogates)
    X = df[FEATURES].values
    scaler = StandardScaler().fit(X)
    X_s = scaler.transform(X)
    
    print("... Training Surrogate Models (CatBoost)")
    m_eui = CatBoostRegressor(iterations=800, verbose=0).fit(X_s, df['EUI'])
    m_idd = CatBoostRegressor(iterations=800, verbose=0).fit(X_s, df['IDD'])
    m_oci = CatBoostRegressor(iterations=800, verbose=0).fit(X_s, df['OCI'])

    # اجرای NSGA-II
    
    problem = BuildingOptimizationProblem([m_eui, m_idd, m_oci], scaler, X.min(axis=0), X.max(axis=0))
    algorithm = NSGA2(pop_size=POP_SIZE, crossover=SBX(prob=0.9, eta=15), mutation=PM(eta=20))
    
    res = minimize(problem, algorithm, get_termination("n_gen", GENS), seed=42, verbose=False)
    
    # رتبه‌بندی نتایج
    final_results = apply_advanced_topsis(res.X, res.F)
    
    # ذخیره فایل اکسل
    final_results.head(50).to_csv(RESULTS_DIR / f"Optimized_{scenario_name}.csv", index=False)
    
    # رسم پلات ۳ بعدی
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    p = ax.scatter(final_results['EUI'], final_results['IDD'], final_results['OCI'], 
                   c=final_results['Performance_Score'], cmap='plasma', s=40)
    
    # علامت‌گذاری بهترین راهکار (Best Compromise)
    best = final_results.iloc[0]
    ax.scatter(best['EUI'], best['IDD'], best['OCI'], color='red', marker='*', s=300, label='Best Solution')
    
    ax.set_xlabel('EUI'); ax.set_ylabel('IDD'); ax.set_zlabel('OCI')
    plt.colorbar(p, label='TOPSIS Performance Score')
    plt.title(f"Pareto Front: {scenario_name}")
    plt.legend()
    plt.savefig(RESULTS_DIR / f"Pareto_{scenario_name}.png", dpi=300)
    plt.close()
    
    return final_results.head(1) # بازگرداندن بهترین نقطه برای مقایسه نهایی

# ==========================================
# 5. اجرای نهایی و مقایسه
# ==========================================
if __name__ == "__main__":
    best_hist = run_full_pipeline("History", FILES["HISTORY"])
    best_future = run_full_pipeline("Future", FILES["FUTURE"])
    
    print("\n✅ All stages completed. Optimized configurations are in the Results folder.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from scipy.interpolate import PchipInterpolator 
import matplotlib.patheffects as pe
from matplotlib import font_manager
from pathlib import Path

# ==========================================
# 1. تنظیمات فونت و استایل (Academic Standard)
# ==========================================
plt.style.use('seaborn-v0_8-whitegrid')
# تلاش برای بارگذاری فونت Times New Roman (در صورت عدم وجود از serif استفاده می‌شود)
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'svg.fonttype': 'none' # برای قابلیت ویرایش متن در ایلاستریتور
})

# ==========================================
# 2. تابع درونیابی برای منحنی‌های نرم
# ==========================================
def get_smooth_curve(x_coords, y_values, n_points=300):
    """ایجاد منحنی نرم با حفظ نقاط عطف اصلی (PCHIP)"""
    pch = PchipInterpolator(x_coords, y_values)
    x_new = np.linspace(min(x_coords), max(x_coords), n_points)
    return x_new, pch(x_new)

# ==========================================
# 3. موتور اصلی ترسیم (The Visual Engine)
# ==========================================
def generate_final_strategy_plot(results_path, output_path):
    print("🎨 Generating Final Strategy Visualization...")
    
    # تنظیم مسیرها
    res_path = Path(results_path)
    out_path = Path(output_path)
    out_path.mkdir(parents=True, exist_ok=True)

    # بارگذاری داده‌های بهینه (Top 50)
    df_hist = pd.read_csv(res_path / "Top50_HISTORY.csv")
    df_fut = pd.read_csv(res_path / "Top50_FUTURE.csv")
    
    # [بخش محاسبات نرمال‌سازی کد شما اینجا قرار می‌گیرد...]
    # ... (کد نرمال‌سازی بالا کاملاً صحیح است)
    
    # تنظیمات رنگی
    colors = {'hist': '#1a535c', 'fut': '#ff6b6b', 'init': '#4ecdc4'}

    fig, ax = plt.subplots(figsize=(22, 11))
    
    # ترسیم ابرهای محدوده (Range Clouds)
    # این بخش نشان‌دهنده فضای عدم قطعیت و گستره پارتو است
    

    # ترسیم استراتژی‌های برتر (Best Curves)
    # اضافه کردن افکت Stroke سفید دور خطوط اصلی برای خوانایی در نقاط تلاقی
    # line_h.set_path_effects([pe.Stroke(linewidth=5, foreground='white'), pe.Normal()])

    # تنظیمات محورها و برچسب‌های درصدی
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['0%', '25%', '50%', '75%', '100%'], fontweight='bold')
    
    # افزودن لیبل پارامترها در پایین با زاویه ۳۰ درجه
    # ax.set_xticklabels(plot_cols, rotation=30, ha='right')

    # ذخیره‌سازی با کیفیت چاپ (300 DPI)
    final_img = out_path / "Strategy_Comparison_Final.png"
    plt.savefig(final_img, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"✅ Final visualization saved to: {final_img}")

# نمونه فراخوانی:
# generate_final_strategy_plot(base_dir, output_folder)